In [0]:
%sql
CREATE OR REPLACE TEMP VIEW geolocation_validated AS
SELECT
  geolocation_zip_code_prefix,
  TRY_CAST(geolocation_lat AS DOUBLE) AS geolocation_lat,
  TRY_CAST(geolocation_lng AS DOUBLE) AS geolocation_lng,
  geolocation_city,
  geolocation_state,
  _ingested_at,
  CASE
    WHEN NOT geolocation_zip_code_prefix RLIKE '^[0-9]{5}$' OR geolocation_zip_code_prefix IS NULL THEN 'zip invalido'
    WHEN TRY_CAST(geolocation_lat AS DOUBLE) IS NULL OR TRY_CAST(geolocation_lng AS DOUBLE) IS NULL THEN 'coordenadas nulas'
    WHEN TRY_CAST(geolocation_lat AS DOUBLE) NOT BETWEEN -34 AND 5.5
      OR TRY_CAST(geolocation_lng AS DOUBLE) NOT BETWEEN -74 AND -34 THEN 'fuera de Brasil'
  END AS motivo
FROM workspace.bronze.geolocation;

CREATE OR REPLACE TABLE workspace.silver.geolocation AS
SELECT
  geolocation_zip_code_prefix,
  AVG(geolocation_lat)    AS geolocation_lat,
  AVG(geolocation_lng)    AS geolocation_lng,
  MODE(geolocation_city)  AS geolocation_city,
  MODE(geolocation_state) AS geolocation_state,
  COUNT(*)                AS geolocation_points
FROM geolocation_validated
WHERE motivo IS NULL
GROUP BY geolocation_zip_code_prefix;

CREATE OR REPLACE TABLE workspace.silver.geolocation_quarantine AS
SELECT * FROM geolocation_validated WHERE motivo IS NOT NULL;